In [1]:
import seaborn
from scipy.special import softmax, log_softmax
from numpy.lib.stride_tricks import sliding_window_view
import scipy.stats as stats
import pickle
import json
from pathlib import Path
import warnings
import torch
import re
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
from seaborn import kdeplot
from tqdm.auto import tqdm
from scipy.stats import gmean
import os
from datetime import datetime

def meanprob(answer_probabilities):
    return gmean([next(iter(value.values())) for value in answer_probabilities])


def is_same_literal_regex(x, y):
    # Remove anything that isn't a letter or a number
    clean_x = re.sub(r'[^a-zA-Z0-9]', '', x).upper()
    clean_y = re.sub(r'[^a-zA-Z0-9]', '', y).upper()

    
    return clean_x == clean_y

/pm/miniconda3/envs/users/han/cot/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
sample_filepaths = {}

sample_filepaths['fake_bootstrap'] = Path("trajectories/test2-fb-llama-bm-thinking-8_llama_regular_bigbench_movie_zero_0_5_type2_conf_fake_bootstrap_hook_mod-none")

In [18]:
confidences = defaultdict(lambda: defaultdict(list))
nb_steps = defaultdict(list)
nb_fake_bootstraps = defaultdict(list)
generated_text = defaultdict(list)
for dataset in sample_filepaths:
    print(f"Loading {dataset}")
    trajectories = []
    sample_files = sorted(sample_filepaths[dataset].glob("traj_*.json"), key=lambda f: int(f.stem.split('_')[1]))  
    #sample_files = [f for f in sample_files if os.path.getmtime(f) > datetime(2026, 4, 14, 17, 3).timestamp()][:60]
    for sample_file in tqdm(sample_files):
        with open(sample_file, "r") as file:
            trajectories += [json.load(file)]

    for trajectory in tqdm(trajectories):
        traj_conf = trajectory['trajectory']['confidence_score']
        if traj_conf is not None:
            perturb_conf = (
                traj_conf.get('fake_bootstrap_hook_confidences')
                # or traj_conf.get('fake_bootstrap_confidences')
                # or traj_conf.get('bootstrap_confidences')
                # or traj_conf.get('jackknife_confidences')
            )
            if perturb_conf is None:
                continue

            generated_text[dataset] += [trajectory['trajectory']['generated_text']]
            nb_steps[dataset] += [len(trajectory['trajectory']['cot_steps'])]
            confidences[dataset]['accuracy'] += [is_same_literal_regex(trajectory['trajectory']['final_answer'], 
                                                                       trajectory['trajectory']['ground_truth'])]
            confidences[dataset]['vanilla_meanprob'] += [meanprob(traj_conf['vanilla_confidences']['answer_probabilities'])]
            confidences[dataset]['fake_bootstrap_meanprob'] += [[meanprob(confidence) for confidence in perturb_conf['answer_probabilities']]]
            confidences[dataset]['vanilla_meanent'] += [np.exp(-meanprob(traj_conf['vanilla_confidences']['answer_entropy']))]
            confidences[dataset]['fake_bootstrap_meanent'] += [[np.exp(-meanprob(confidence)) for confidence in perturb_conf['answer_entropy']]]
            confidences[dataset]['vanilla_indirect1'] += [traj_conf['vanilla_confidences']['indirect_ptrue1_probabilities'][0]['True']]
            confidences[dataset]['fake_bootstrap_indirect1'] += [[confidence['True'] for confidence in perturb_conf['indirect_ptrue1_probabilities']]]
            confidences[dataset]['vanilla_indirect2'] += [traj_conf['vanilla_confidences']['indirect_ptrue2_probabilities'][0]['Yes']]
            confidences[dataset]['fake_bootstrap_indirect2'] += [[confidence['Yes'] for confidence in perturb_conf['indirect_ptrue2_probabilities']]]
            confidences[dataset]['vanilla_indirect3'] += [traj_conf['vanilla_confidences']['indirect_ptrue3_probabilities'][0]['Yes']]
            confidences[dataset]['fake_bootstrap_indirect3'] += [[confidence['Yes'] for confidence in perturb_conf['indirect_ptrue3_probabilities']]]
            confidences[dataset]['vanilla_verbal'] += [traj_conf['vanilla_confidences']['verbconf_probabilities'][0]]
            confidences[dataset]['fake_bootstrap_verbal'] += [perturb_conf['verbconf_probabilities']]

    nb_steps[dataset] = torch.LongTensor(nb_steps[dataset])
    nb_fake_bootstraps[dataset] = torch.ceil(torch.sqrt(nb_steps[dataset]))
    for key in confidences[dataset]:
        confidences[dataset][key] = torch.FloatTensor(confidences[dataset][key])

Loading fake_bootstrap


100%|██████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 30.62it/s]


In [19]:
dataset = 'fake_bootstrap'
confidences['fake_bootstrap'].keys()


dict_keys(['accuracy', 'vanilla_meanprob', 'fake_bootstrap_meanprob', 'vanilla_meanent', 'fake_bootstrap_meanent', 'vanilla_indirect1', 'fake_bootstrap_indirect1', 'vanilla_indirect2', 'fake_bootstrap_indirect2', 'vanilla_indirect3', 'fake_bootstrap_indirect3', 'vanilla_verbal', 'fake_bootstrap_verbal'])

In [20]:
confidences[dataset]['vanilla_meanprob']

tensor([0.9701, 0.9881, 0.6834, 0.9184, 0.9961])

In [21]:
confidences[dataset]['fake_bootstrap_meanprob']


tensor([[0.9697, 0.9727, 0.9697, 0.9730, 0.9697, 0.9697, 0.9697, 0.9727, 0.9730,
         0.9730, 0.9730, 0.9730, 0.9730, 0.9727, 0.9697, 0.9727, 0.9730, 0.9727,
         0.9730, 0.9727, 0.9730, 0.9730, 0.9727, 0.9697, 0.9730, 0.9697, 0.9697,
         0.9730, 0.9727, 0.9730, 0.9730, 0.9697, 0.9730, 0.9697, 0.9697, 0.9727,
         0.9727, 0.9697, 0.9730, 0.9730, 0.9730, 0.9730, 0.9730, 0.9697, 0.9730,
         0.9727, 0.9730, 0.9730, 0.9730, 0.9730, 0.9697, 0.9697, 0.9697, 0.9727,
         0.9727, 0.9697, 0.9697, 0.9730, 0.9730, 0.9727, 0.9730, 0.9697, 0.9697,
         0.9727, 0.9727, 0.9730, 0.9697, 0.9730, 0.9697, 0.9730, 0.9730, 0.9730,
         0.9730, 0.9730, 0.9730, 0.9727, 0.9730, 0.9727, 0.9727, 0.9730, 0.9730,
         0.9697, 0.9730, 0.9730, 0.9730, 0.9727, 0.9697, 0.9730, 0.9697, 0.9727,
         0.9730, 0.9727, 0.9727, 0.9730, 0.9730, 0.9730, 0.9730, 0.9730, 0.9730,
         0.9730],
        [0.9859, 0.9886, 0.9883, 0.9875, 0.9865, 0.9874, 0.9894, 0.9889, 0.9888,
         0

In [22]:
confidences[dataset]['vanilla_indirect1']

tensor([0.7311, 0.7186, 0.6654, 0.7607, 0.7058])

In [24]:
confidences[dataset]['fake_bootstrap_indirect1']

tensor([[0.7311, 0.7249, 0.7311, 0.7249, 0.7311, 0.7311, 0.7311, 0.7249, 0.7249,
         0.7249, 0.7249, 0.7249, 0.7249, 0.7249, 0.7311, 0.7249, 0.7249, 0.7249,
         0.7249, 0.7249, 0.7249, 0.7249, 0.7249, 0.7311, 0.7249, 0.7311, 0.7311,
         0.7249, 0.7249, 0.7249, 0.7249, 0.7311, 0.7249, 0.7311, 0.7311, 0.7249,
         0.7249, 0.7311, 0.7249, 0.7249, 0.7249, 0.7249, 0.7249, 0.7311, 0.7249,
         0.7249, 0.7249, 0.7249, 0.7249, 0.7249, 0.7311, 0.7311, 0.7311, 0.7249,
         0.7249, 0.7311, 0.7311, 0.7249, 0.7249, 0.7249, 0.7249, 0.7311, 0.7311,
         0.7249, 0.7249, 0.7249, 0.7311, 0.7249, 0.7311, 0.7249, 0.7249, 0.7249,
         0.7249, 0.7249, 0.7249, 0.7249, 0.7249, 0.7249, 0.7249, 0.7249, 0.7249,
         0.7311, 0.7249, 0.7249, 0.7249, 0.7249, 0.7311, 0.7249, 0.7311, 0.7249,
         0.7249, 0.7249, 0.7249, 0.7249, 0.7249, 0.7249, 0.7249, 0.7249, 0.7249,
         0.7249],
        [0.7311, 0.7311, 0.7311, 0.7311, 0.7186, 0.7186, 0.7311, 0.7186, 0.7311,
         0

In [26]:
confidences[dataset]['vanilla_verbal']

tensor([0.4569, 0.8587, 0.3248, 0.8234, 0.7720])

In [25]:
confidences[dataset]['fake_bootstrap_verbal']

tensor([[0.4514, 0.4521, 0.4514, 0.4542, 0.4514, 0.4514, 0.4514, 0.4521, 0.4542,
         0.4542, 0.4542, 0.4542, 0.4542, 0.4521, 0.4514, 0.4521, 0.4542, 0.4521,
         0.4542, 0.4521, 0.4542, 0.4542, 0.4521, 0.4514, 0.4542, 0.4514, 0.4514,
         0.4542, 0.4521, 0.4542, 0.4542, 0.4514, 0.4542, 0.4514, 0.4514, 0.4521,
         0.4521, 0.4514, 0.4542, 0.4542, 0.4542, 0.4542, 0.4542, 0.4514, 0.4542,
         0.4521, 0.4542, 0.4542, 0.4542, 0.4542, 0.4514, 0.4514, 0.4514, 0.4521,
         0.4521, 0.4514, 0.4514, 0.4542, 0.4542, 0.4521, 0.4542, 0.4514, 0.4514,
         0.4521, 0.4521, 0.4542, 0.4514, 0.4542, 0.4514, 0.4542, 0.4542, 0.4542,
         0.4542, 0.4542, 0.4542, 0.4521, 0.4542, 0.4521, 0.4521, 0.4542, 0.4542,
         0.4514, 0.4542, 0.4542, 0.4542, 0.4521, 0.4514, 0.4542, 0.4514, 0.4521,
         0.4542, 0.4521, 0.4521, 0.4542, 0.4542, 0.4542, 0.4542, 0.4542, 0.4542,
         0.4542],
        [0.8521, 0.8508, 0.8546, 0.8553, 0.8547, 0.8557, 0.8541, 0.8536, 0.8568,
         0

In [19]:
filepath = Path("/storage/backup/han/backup_workspace/cot-eval/trajectories/1_true_bootstrap/bootstrap-bm-428-llama_llama_regular_bigbench_movie_zero_0_250_type2_conf_bootstrap_mod-none/traj_1.json")

In [20]:

with open(filepath, "r") as file:
    temp = [json.load(file)]

In [21]:
temp[0].keys()

dict_keys(['index', 'question', 'trajectory'])

In [24]:
all_prompts = temp[0]['trajectory']['debug_info']['all_prompts']

In [27]:
all_prompts.keys()

dict_keys(['answer_logits', 'indirect_ptrue1', 'indirect_ptrue2', 'verbconf'])

In [26]:
print(all_prompts['answer_logits'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are an expert reasoning assistant. For every problem you receive, think carefully and reason step-by-step. Label each reasoning step as 'Step 1:', 'Step 2:', etc. Keep your reasoning concise, and be brief in each step.

When you output the final answer, output ONLY the single letter corresponding to the correct answer. Do not include periods, explanations, or any other text.

During your reasoning, do NOT reveal, hint at, or restate the final answer. Do not write lines like 'Answer:', 'Final answer:', any answer strings, or any concluding sentence. Stop immediately after your last numbered reasoning step.<|eot_id|><|start_header_id|>user<|end_header_id|>

Find a movie similar to The Sixth Sense, The Matrix, Forrest Gump, The Shawshank Redemption:
Options:
(A) Street Fighter II The Animated Movie
(B) The Sheltering Sky
(C) The Boy Who Could Fly
(D) Terminator 2

In [28]:
print(all_prompts['indirect_ptrue1'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are an expert reasoning assistant. For every problem you receive, think carefully and reason step-by-step. Label each reasoning step as 'Step 1:', 'Step 2:', etc. Keep your reasoning concise, and be brief in each step.

When you output the final answer, output ONLY the single letter corresponding to the correct answer. Do not include periods, explanations, or any other text.

During your reasoning, do NOT reveal, hint at, or restate the final answer. Do not write lines like 'Answer:', 'Final answer:', any answer strings, or any concluding sentence. Stop immediately after your last numbered reasoning step.<|eot_id|><|start_header_id|>user<|end_header_id|>

Find a movie similar to The Sixth Sense, The Matrix, Forrest Gump, The Shawshank Redemption:
Options:
(A) Street Fighter II The Animated Movie
(B) The Sheltering Sky
(C) The Boy Who Could Fly
(D) Terminator 2

In [29]:
print(all_prompts['verbconf'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are an expert reasoning assistant. For every problem you receive, think carefully and reason step-by-step. Label each reasoning step as 'Step 1:', 'Step 2:', etc. Keep your reasoning concise, and be brief in each step.

When you output the final answer, output ONLY the single letter corresponding to the correct answer. Do not include periods, explanations, or any other text.

During your reasoning, do NOT reveal, hint at, or restate the final answer. Do not write lines like 'Answer:', 'Final answer:', any answer strings, or any concluding sentence. Stop immediately after your last numbered reasoning step.<|eot_id|><|start_header_id|>user<|end_header_id|>

Find a movie similar to The Sixth Sense, The Matrix, Forrest Gump, The Shawshank Redemption:
Options:
(A) Street Fighter II The Animated Movie
(B) The Sheltering Sky
(C) The Boy Who Could Fly
(D) Terminator 2

In [34]:
bootstrap_prompts = temp[0]['trajectory']['debug_info']['bootstrap_prompts']

In [35]:
bootstrap_prompts.keys()

dict_keys(['answer_logits', 'indirect_ptrue1', 'indirect_ptrue2', 'verbconf'])

In [32]:
print(bootstrap_prompts['answer_logits'][0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are an expert reasoning assistant. For every problem you receive, think carefully and reason step-by-step. Label each reasoning step as 'Step 1:', 'Step 2:', etc. Keep your reasoning concise, and be brief in each step.

When you output the final answer, output ONLY the single letter corresponding to the correct answer. Do not include periods, explanations, or any other text.

During your reasoning, do NOT reveal, hint at, or restate the final answer. Do not write lines like 'Answer:', 'Final answer:', any answer strings, or any concluding sentence. Stop immediately after your last numbered reasoning step.<|eot_id|><|start_header_id|>user<|end_header_id|>

Find a movie similar to The Sixth Sense, The Matrix, Forrest Gump, The Shawshank Redemption:
Options:
(A) Street Fighter II The Animated Movie
(B) The Sheltering Sky
(C) The Boy Who Could Fly
(D) Terminator 2

In [33]:
print(bootstrap_prompts['indirect_ptrue1'][0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are an expert reasoning assistant. For every problem you receive, think carefully and reason step-by-step. Label each reasoning step as 'Step 1:', 'Step 2:', etc. Keep your reasoning concise, and be brief in each step.

When you output the final answer, output ONLY the single letter corresponding to the correct answer. Do not include periods, explanations, or any other text.

During your reasoning, do NOT reveal, hint at, or restate the final answer. Do not write lines like 'Answer:', 'Final answer:', any answer strings, or any concluding sentence. Stop immediately after your last numbered reasoning step.<|eot_id|><|start_header_id|>user<|end_header_id|>

Find a movie similar to The Sixth Sense, The Matrix, Forrest Gump, The Shawshank Redemption:
Options:
(A) Street Fighter II The Animated Movie
(B) The Sheltering Sky
(C) The Boy Who Could Fly
(D) Terminator 2

In [36]:
print(bootstrap_prompts['verbconf'][0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are an expert reasoning assistant. For every problem you receive, think carefully and reason step-by-step. Label each reasoning step as 'Step 1:', 'Step 2:', etc. Keep your reasoning concise, and be brief in each step.

When you output the final answer, output ONLY the single letter corresponding to the correct answer. Do not include periods, explanations, or any other text.

During your reasoning, do NOT reveal, hint at, or restate the final answer. Do not write lines like 'Answer:', 'Final answer:', any answer strings, or any concluding sentence. Stop immediately after your last numbered reasoning step.<|eot_id|><|start_header_id|>user<|end_header_id|>

Find a movie similar to The Sixth Sense, The Matrix, Forrest Gump, The Shawshank Redemption:
Options:
(A) Street Fighter II The Animated Movie
(B) The Sheltering Sky
(C) The Boy Who Could Fly
(D) Terminator 2